# Environment / Dependencies Setup
The cell below installs all required Python packages:

In [ ]:
!pip install -qU pypdf neo4j sentence-transformers "torch>=2.5" langchain-huggingface langchain-openai yfiles-jupyter-graphs-for-neo4j python-dotenv

## Import Libraries

In [ ]:
import os
import json
import requests
from dotenv import load_dotenv
from pypdf import PdfReader
from neo4j import GraphDatabase
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import PromptTemplate

# Load the secrets (OPENROUTER_API_KEY, NEO4J_*) from the .env file next to this notebook
load_dotenv(".env")


## Connect to Neo4j

In [ ]:
# --- Neo4j Aura Configuration (values come from the .env file) ---
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USER = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

# Open a persistent connection to the database
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
# fails fast here if the uri or credentials are wrong
driver.verify_connectivity()
print("Success: Connected to Neo4j Database!")


## Configure the LLM

In [ ]:
llm = ChatOpenAI(
    openai_api_key=os.getenv("OPENROUTER_API_KEY"),
    openai_api_base="https://openrouter.ai/api/v1",
    model_name="nvidia/nemotron-3-super-120b-a12b:free",
    temperature=0.0,
    max_tokens=8000,
    # openrouter-specific flag: disable the model's reasoning output
    extra_body={"reasoning": {"enabled": False}},
)
print("LangChain LLM configured successfully!")


### Step 1 — Download PDF from Link & Extract Text

In [ ]:
os.makedirs("data", exist_ok=True)

PDF_URL = "https://arxiv.org/pdf/1706.03762.pdf"
PDF_FILENAME = "data/downloaded_paper.pdf"

print(f"Downloading PDF from link: {PDF_URL}...")
response = requests.get(PDF_URL)
response.raise_for_status()

with open(PDF_FILENAME, "wb") as f:
    f.write(response.content)
print(f"PDF downloaded successfully to {PDF_FILENAME}!")

print("Extracting text from PDF...")
reader = PdfReader(PDF_FILENAME)
document_text = ""

for page in reader.pages:
    document_text += page.extract_text() + "\n"
    # Stop after 1500 characters, so the demo runs fast
    if len(document_text) >= 1500:
        document_text = document_text[:1500]
        break

print(f"Extracted {len(document_text)} characters of text.")

### Step 2 — Extract Graph Structure
The extraction prompt goes up first, with the JSON shape spelled out:

In [ ]:
print("Asking LLM to extract graph nodes and edges...")

prompt = f"""
You are a data extraction AI. Read the text and extract a knowledge graph.
Identify key concepts and the relationships between them.

TEXT:
{document_text}

CRITICAL INSTRUCTION: Reply ONLY with valid JSON.
Format exactly like this:
{{
  "nodes": [ {{"name": "Concept 1"}}, {{"name": "Concept 2"}} ],
  "relationships": [ {{"source": "Concept 1", "target": "Concept 2", "type": "RELATES_TO"}} ]
}}
"""

Then the call, plus the strip of any Markdown fence the model wrapped its JSON in:

In [ ]:
response = llm.invoke(prompt)

raw_output = response.content.strip()
if raw_output.startswith("```json"):
    # slice off the markdown fence: 7 chars in front, 3 at the end
    raw_output = raw_output[7:-3].strip()
elif raw_output.startswith("```"):
    raw_output = raw_output[3:-3].strip()

graph_data = json.loads(raw_output)
print(f"Extracted {len(graph_data.get('nodes', []))} nodes and {len(graph_data.get('relationships', []))} relationships!")

### Step 3 — Save Graph to Neo4j (Raw Cypher)
Nodes are written first, then relationships, since a relationship can only be created between nodes that already exist.

In [ ]:
# Save nodes to Neo4j
# merge creates the concept only if that name does not exist yet, reruns stay safe
node_query = """
UNWIND $nodes AS node
MERGE (c:Concept {name: node.name})
"""

with driver.session() as session:
    session.run(node_query, nodes=graph_data.get("nodes", []))

print("Nodes saved to Neo4j!")

In [ ]:
# Save relationships to Neo4j
rel_query = """
UNWIND $relationships AS rel
MATCH (source:Concept {name: rel.source})
MATCH (target:Concept {name: rel.target})
CALL apoc.create.relationship(source, replace(toUpper(rel.type), ' ', '_'), {}, target) YIELD rel AS r
RETURN count(r)
"""
# This special command lets us name the relationship using a variable (like "BASED_ON").
# Normal Cypher code cannot do this.

with driver.session() as session:
    if graph_data.get("relationships"):
        session.run(rel_query, relationships=graph_data.get("relationships", []))

print("Relationships saved to Neo4j!")

### Step 4 — Initialize Embedding Model & Create Vector Index

In [ ]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# This model always makes vectors of size 384. The number below must match.
index_query = """
CREATE VECTOR INDEX `concept_embeddings` IF NOT EXISTS 
FOR (c:Concept) ON (c.embedding) 
OPTIONS {
    indexConfig: { 
        `vector.dimensions`: 384, 
        `vector.similarity_function`: 'cosine' 
    }
}
"""

with driver.session() as session:
    session.run(index_query)
print("Vector index 'concept_embeddings' is ready.")

### Step 5 — Generate and Store Vector Embeddings

In [ ]:
with driver.session() as session:
    # Only get nodes that do not have a vector yet, so we don't do the work twice
    result = session.run("MATCH (c:Concept) WHERE c.embedding IS NULL RETURN elementId(c) as node_id, c.name AS text")
    records = [record for record in result]
    
    if not records:
        print("All nodes already have embeddings.")
    else:
        print(f"Generating vectors for {len(records)} nodes...")
        for record in records:
            vector = embeddings.embed_query(record["text"])
            session.run(
                "MATCH (c) WHERE elementId(c) = $node_id SET c.embedding = $vector", 
                node_id=record["node_id"], 
                vector=vector
            )
        print("Success: All graph concepts vectorized!")

### Step 6 — Define the Hybrid Retrieval Function

In [ ]:
def execute_hybrid_retrieval(driver, question, top_k=3):
    print(f"User Question: '{question}'\n")

    question_vector = embeddings.embed_query(question)

    # vector search finds the seed nodes, then the match expands to their neighbors
    hybrid_query = """
    CALL db.index.vector.queryNodes('concept_embeddings', $top_k, $question_vector)
    YIELD node AS seed, score
    MATCH (seed)-[r]-(neighbor:Concept)
    RETURN seed.name AS Seed_Node, score AS Semantic_Score, startNode(r).name AS Source, type(r) AS Relationship, endNode(r).name AS Target
    """

    retrieved_facts = []
    with driver.session() as session:
        result = session.run(hybrid_query, top_k=top_k, question_vector=question_vector)

        print("RETRIEVED HYBRID PATHS:")
        print("-" * 60)
        for record in result:
            fact = f"(Similarity: {record['Semantic_Score']:.2f}) {record['Source']} --[{record['Relationship']}]--> {record['Target']}"
            retrieved_facts.append(fact)
            print(fact)

    return retrieved_facts

### Step 7 — Generate Answer using LangChain Prompt Pipelines
The answer prompt is defined first — two blank spots, one for the retrieved facts and one for the question:

In [ ]:
ANSWER_TEMPLATE = """
You are an expert AI research assistant using a Neo4j Knowledge Graph.
Answer the question using ONLY the connected relationship paths provided below.

Graph Relationships:
{facts_block}

Question: {question}

Output your response in EXACTLY two sections:
--- FINAL ANSWER ---
[Provide a direct, simple, 1-sentence answer.]

--- AI TRACING & EXPLAINABILITY ---
[Explain step-by-step how the answer was derived from the Neo4j graph context.]
"""

Then the function that fills those blanks and chains the template straight to the LLM:

In [ ]:
def generate_hybrid_answer(question, retrieved_facts):
    if not retrieved_facts:
        return "No relevant context found in the database."

    facts_block = "\n".join([f"- {f}" for f in retrieved_facts])
    prompt = PromptTemplate(template=ANSWER_TEMPLATE, input_variables=["facts_block", "question"])
    # | pipes the filled template into the model, langchain chain syntax
    chain = prompt | llm

    response = chain.invoke({"facts_block": facts_block, "question": question})
    return response.content

### Step 8 — Querying

In [ ]:
query = "What models or mechanisms are discussed in the document?"

facts = execute_hybrid_retrieval(driver, query, top_k=3)

print(generate_hybrid_answer(query, facts))

### Step 9 — Visualize the Graph Inside the Notebook

In [ ]:
from yfiles_jupyter_graphs_for_neo4j import Neo4jGraphWidget

widget = Neo4jGraphWidget(driver)
widget.show_cypher("MATCH (n:Concept)-[r]-(m:Concept) RETURN n, r, m")